# 07 Nonlinear Time Series and Particle Filters

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/08-Time-Series/07_Nonlinear_Time_Series_and_Particle_Filters.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=08-Time-Series/07_Nonlinear_Time_Series_and_Particle_Filters.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: Filtering When Gaussian Linearity Breaks
The Kalman filter is exact because linear state transitions and Gaussian shocks preserve a Gaussian filtering distribution. Many economic state-space models violate both assumptions: volatility is positive and nonlinear, regime changes are discrete, occasionally binding constraints kink policy rules, and measurement equations can be strongly non-Gaussian. Sequential Monte Carlo (SMC), usually called a particle filter in this setting, replaces a single Gaussian approximation with a weighted empirical distribution of simulated states.

The economic question is whether the hidden state can be learned from the sequence of observations without numerical collapse. A particle filter can fail even when the code runs: weights may concentrate on one particle, resampling may destroy diversity, or a likelihood may underflow to zero. This notebook therefore treats log-weight stabilization and effective sample size (ESS) as first-class diagnostics. We simulate a canonical nonlinear state-space process, implement a bootstrap filter, trigger systematic resampling adaptively, and compare the filtered-state RMSE with a naive observation-based proxy.

### Learning Objectives
- **Derive** the sequential importance-weighting recursion.
- **Implement** a numerically stable bootstrap particle filter with systematic resampling.
- **Diagnose** particle degeneracy using ESS and log-likelihood increments.
- **Connect** SMC to nonlinear DSGE, stochastic-volatility, and regime-switching applications.

### Prerequisites
- `01_Introduction_to_Time_Series.ipynb`: state-space representation and innovations.
- `../02-Numerical-Methods/02_Numerical_Preliminaries.ipynb`: log-sum-exp and floating-point stability.
- Probability distributions and Monte Carlo simulation.
* **Learning-path prerequisite:** [`06_Cointegration_and_Error_Correction_Models.ipynb`](06_Cointegration_and_Error_Correction_Models.ipynb)


> **Learning path:** Building on [`06_Cointegration_and_Error_Correction_Models.ipynb`](06_Cointegration_and_Error_Correction_Models.ipynb); this notebook closes the current track.


## Table of Contents

1. [Nonlinear state-space model](#nonlinear-state-space)
2. [Sequential importance sampling](#sequential-importance)
3. [Systematic resampling](#systematic-resampling)
4. [Bootstrap particle filter](#bootstrap-filter)
5. [Diagnostics](#diagnostics)
6. [Exercises](#exercises)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 12, "figure.figsize": (10, 6), "figure.dpi": 120})
np.set_printoptions(suppress=True, precision=5, linewidth=120)

from scipy.special import logsumexp


<a id="nonlinear-state-space"></a>
## 1. Nonlinear State-Space Model

We use the standard nonlinear filtering benchmark

$$x_t=\frac{x_{t-1}}{2}+\frac{25x_{t-1}}{1+x_{t-1}^2}+8\cos(1.2t)+\eta_t,$$

$$y_t=\frac{x_t^2}{20}+\varepsilon_t,$$

with Gaussian state and measurement shocks. The observation equation is many-to-one because $x$ and $-x$ imply the same conditional mean. A Gaussian linearization can therefore be seriously misleading.


In [ ]:
def simulate_nonlinear_state_space(n=80, state_sd=np.sqrt(10.0), obs_sd=1.0, seed=7):
    local_rng = np.random.default_rng(seed)
    x = np.zeros(n)
    y = np.zeros(n)
    x[0] = local_rng.normal(scale=np.sqrt(5.0))
    y[0] = x[0] ** 2 / 20 + local_rng.normal(scale=obs_sd)
    for t in range(1, n):
        x_prev = x[t - 1]
        mean = 0.5 * x_prev + 25 * x_prev / (1 + x_prev**2) + 8 * np.cos(1.2 * t)
        x[t] = mean + local_rng.normal(scale=state_sd)
        y[t] = x[t] ** 2 / 20 + local_rng.normal(scale=obs_sd)
    return x, y

true_state, observations = simulate_nonlinear_state_space()


<a id="sequential-importance"></a>
## 2. Sequential Importance Sampling

For particles $x_t^{(i)}$ proposed from the transition density, bootstrap-filter weights are proportional to the observation likelihood:

$$\tilde w_t^{(i)} = w_{t-1}^{(i)}p(y_t\mid x_t^{(i)}),\qquad
w_t^{(i)}=\frac{\tilde w_t^{(i)}}{\sum_j\tilde w_t^{(j)}}.$$

Products of small likelihoods underflow quickly, so we work in log space and normalize with

$$\log\sum_i e^{a_i}=m+\log\sum_i e^{a_i-m},\qquad m=\max_i a_i.$$

The effective sample size

$$ESS_t=\frac{1}{\sum_i(w_t^{(i)})^2}$$

summarizes weight concentration. We resample only when ESS falls below a fraction of the particle count.


<a id="systematic-resampling"></a>
## 3. Systematic Resampling

Systematic resampling uses one uniform draw and evenly spaced cumulative-probability cutoffs. It has lower Monte Carlo variance than independent multinomial resampling while remaining simple and $O(N)$.


In [ ]:
def systematic_resample(weights, rng):
    n = len(weights)
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    indices = np.searchsorted(cumulative, positions, side="right")
    return np.minimum(indices, n - 1)


def log_normal_pdf(x, mean, sd):
    z = (x - mean) / sd
    return -0.5 * z**2 - np.log(sd) - 0.5 * np.log(2 * np.pi)


<a id="bootstrap-filter"></a>
## 4. Bootstrap Particle Filter


In [ ]:
def bootstrap_particle_filter(y, n_particles=4_000, state_sd=np.sqrt(10.0), obs_sd=1.0, ess_fraction=0.5, seed=99):
    pf_rng = np.random.default_rng(seed)
    particles = pf_rng.normal(scale=np.sqrt(5.0), size=n_particles)
    log_weights = np.full(n_particles, -np.log(n_particles))
    filtered = np.empty(len(y))
    ess_path = np.empty(len(y))
    loglik = 0.0

    for t, obs in enumerate(y):
        if t > 0:
            mean = 0.5 * particles + 25 * particles / (1 + particles**2) + 8 * np.cos(1.2 * t)
            particles = mean + pf_rng.normal(scale=state_sd, size=n_particles)

        observation_mean = particles**2 / 20
        log_increment = log_normal_pdf(obs, observation_mean, obs_sd)
        log_unnormalized = log_weights + log_increment
        norm = logsumexp(log_unnormalized)
        log_weights = log_unnormalized - norm
        weights = np.exp(log_weights)
        loglik += norm

        filtered[t] = np.sum(weights * particles)
        ess = 1.0 / np.sum(weights**2)
        ess_path[t] = ess
        if ess < ess_fraction * n_particles:
            indices = systematic_resample(weights, pf_rng)
            particles = particles[indices]
            log_weights.fill(-np.log(n_particles))

    return filtered, ess_path, float(loglik)

filtered_state, ess_path, particle_loglik = bootstrap_particle_filter(observations)
rmse = float(np.sqrt(np.mean((filtered_state - true_state) ** 2)))
naive_proxy = np.sqrt(np.maximum(20 * observations, 0))
naive_rmse = float(np.sqrt(np.mean((naive_proxy - np.abs(true_state)) ** 2)))
print(f"particle-filter state RMSE = {rmse:.3f}")
print(f"naive |state| proxy RMSE = {naive_rmse:.3f}")
print(f"minimum ESS = {ess_path.min():.0f}; log likelihood = {particle_loglik:.2f}")
assert np.all(np.isfinite(filtered_state)) and np.all(ess_path > 0)


<a id="diagnostics"></a>
## 5. Diagnostics

The filtered mean can still be a poor summary of a multimodal posterior, so an applied workflow should retain particle quantiles or the full cloud when sign ambiguity matters. ESS reveals degeneracy but not model misspecification; residual or predictive checks are still required.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes[0].plot(true_state, label="latent state", lw=2)
axes[0].plot(filtered_state, label="particle-filter mean", alpha=0.8)
axes[0].set(ylabel="state", title="Nonlinear filtering")
axes[0].legend()
axes[1].plot(ess_path)
axes[1].axhline(0.5 * 4_000, color="black", ls="--", label="resampling threshold")
axes[1].set(xlabel="time", ylabel="ESS")
axes[1].legend()
plt.show()


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$x_t=\frac{x_{t-1}}{2}+\frac{25x_{t-1}}{1+x_{t-1}^2}+8\cos(1.2t)+\eta_t,$$

**2. Core relation**

$$y_t=\frac{x_t^2}{20}+\varepsilon_t,$$

**3. Core relation**

$$\tilde w_t^{(i)} = w_{t-1}^{(i)}p(y_t\mid x_t^{(i)}),\qquad w_t^{(i)}=\frac{\tilde w_t^{(i)}}{\sum_j\tilde w_t^{(j)}}.$$

**4. Core relation**

$$\log\sum_i e^{a_i}=m+\log\sum_i e^{a_i-m},\qquad m=\max_i a_i.$$


## Exercises

**1. Degeneracy (Conceptual):** Show that equal weights imply `ESS=N` and a single particle with weight one implies `ESS=1`. Explain why resampling every period can also be harmful.

**2. Particle count (Applied):** Re-run the filter with 250, 1,000, 4,000, and 16,000 particles across 20 seeds. Report RMSE, log-likelihood variability, and runtime; identify diminishing returns.

**3. Economic state-space model (Challenge):** Replace the benchmark transition with a stochastic-volatility or nonlinear DSGE state equation. Write the exact proposal and weight equations before adapting the code.


## Economic Interpretation: Filtering is real-time inference

A particle filter converts a nonlinear state-space model into a sequence of empirical posterior distributions. In macro and finance the latent state can represent volatility, a regime, productivity, or another economically meaningful object that is never observed directly. The resampling step concentrates computational effort on state paths that remain plausible after new data arrive, so filtering uncertainty is part of the economic inference rather than a numerical nuisance to hide.


## Summary & Key Takeaways

- Particle filters approximate nonlinear/non-Gaussian filtering distributions with weighted simulated states.
- Log-weight normalization is mandatory numerical hygiene, not an optimization trick.
- Effective sample size identifies degeneracy and supports adaptive resampling.
- Filtering accuracy and model adequacy are distinct questions; both require diagnostics.


## References & Further Reading

- Gordon, N. J., Salmond, D. J. & Smith, A. F. M. (1993). Novel approach to nonlinear/non-Gaussian Bayesian state estimation. *IEE Proceedings F*, 140(2), 107–113.
- Doucet, A. & Johansen, A. M. (2011). A tutorial on particle filtering and smoothing. In *The Oxford Handbook of Nonlinear Filtering*.
- Herbst, E. P. & Schorfheide, F. (2015). *Bayesian Estimation of DSGE Models*. Princeton University Press.
